In [ ]:
"""

Outputs:
  figures/
    fig_A_AUC_trajectory.pdf
    fig_B_preclinical_timing.pdf
    fig_C_sensitivity_by_onset.pdf
  tables/
    A_window_iAUC.csv
    A_retention_ratio.csv
    B_top10_timing.csv
    C_sensitivity_by_onset.csv
"""

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from joblib import Parallel, delayed

from sksurv.util import Surv
from sksurv.metrics import cumulative_dynamic_auc
from sksurv.nonparametric import kaplan_meier_estimator
from statsmodels.stats.multitest import multipletests


# ========================= Configuration =========================

PROJECT_DIR = Path("./data/work")
DATA_PATH   = PROJECT_DIR / "BrainVital8_all_cohorts.csv"

OUT_DIR = "./results/preclinical_identification"
FIG_DIR = OUT_DIR / "figures"
CSV_DIR = OUT_DIR / "tables"
for d in [FIG_DIR, CSV_DIR]:
    d.mkdir(parents=True, exist_ok=True)

time_col   = "time"
event_col  = "status"
cohort_col = "cohort"

MAIN_COL   = "BrainVital8"
MAIN_LABEL = "BrainVital8"

COMPARATORS = [
    ("Age",      "age"),
    ("LIBRA2",   "libra2_0"),
    ("CAIDE",    "caide_0_noage"),
    ("ANU-ADRI", "ANU_0_noage"),
    ("CogDrisk", "CogD_0_noage"),
    ("LE8",      "le8_0"),
]

# Scores where higher = healthier -> negate for risk direction
NEGATE_SCORES = {MAIN_COL, "le8_0"}

# Time windows: near-term vs long-term (preclinical)
TIME_WINDOWS = [
    ("0-3y",  0,  3),
    ("3-5y",  3,  5),
    ("5-10y", 5, 10),
    ("10y+", 10, None),
]

# Early vs late onset cutoffs (years)
EARLY_ONSET_CUTOFF = 3
LATE_ONSET_CUTOFF  = 5

COHORT_GRID = {
    "UKB":   dict(q_end=0.95, n_per_window=10),
    "ELSA":  dict(q_end=0.70, n_per_window=8),
    "SHARE": dict(q_end=0.95, n_per_window=10),
    "HRS":   dict(q_end=0.95, n_per_window=10),
}
DEFAULT_GRID = dict(q_end=0.95, n_per_window=10)

TOP_QUANTILES = [0.10, 0.20]
N_BOOT = 1000
RANDOM_SEED = 42
N_JOBS = -1

COLORS = {
    "BrainVital8": "#E63946",
    "Age":         "#6C757D",
    "LIBRA2":      "#457B9D",
    "CAIDE":       "#2A9D8F",
    "ANU-ADRI":    "#8FB8A0",
    "CogDrisk":    "#F4A261",
    "LE8":         "#1D3557",
}


# ========================= Utilities =========================

def coerce_numeric(df, cols):
    """Convert columns to numeric, coercing errors to NaN."""
    for c in cols:
        if c in df.columns:
            if df[c].dtype == "object":
                df[c] = df[c].astype(str).str.strip()
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def risk_array(df, col):
    """Get risk array, negating scores where higher = healthier."""
    raw = df[col].to_numpy(dtype=float)
    return -raw if col in NEGATE_SCORES else raw


def get_aligned(df_in, score_cols):
    """Drop rows with missing scores/time/event, keep time > 0."""
    valid = [c for c in score_cols if c in df_in.columns]
    sub = df_in.dropna(subset=valid + [time_col, event_col]).copy()
    sub = sub[sub[time_col] > 0]
    return sub, valid


# ========================= Module A: iAUC Trajectory =========================

def compute_window_iauc(aligned, score_col, t_start_d, t_end_d, n_pts):
    """Compute integrated AUC within a single time window."""
    time_arr  = aligned[time_col].to_numpy(dtype=float)
    event_arr = aligned[event_col].to_numpy(dtype=int).astype(bool)
    risk      = risk_array(aligned, score_col)
    t_start_d = max(30.0, t_start_d)
    if t_end_d <= t_start_d:
        return np.nan
    times = np.linspace(t_start_d, t_end_d, n_pts)
    try:
        y = Surv.from_arrays(event=event_arr, time=time_arr)
        _, iauc = cumulative_dynamic_auc(y, y, risk, times)
        return float(iauc)
    except Exception:
        return np.nan


def paired_retention_ratio(aligned, score_a, score_b,
                           t_early_end_d, t_late_start_d, t_late_end_d,
                           n_pts, n_boot, seed, n_jobs):
    """Compute retention ratio (late iAUC / early iAUC) for two scores,
    with paired bootstrap for delta and 95% CI.
    Ratio > 1 means discrimination does not decay over time.
    """
    time_arr  = aligned[time_col].to_numpy(dtype=float)
    event_arr = aligned[event_col].to_numpy(dtype=int).astype(bool)
    a_risk    = risk_array(aligned, score_a)
    b_risk    = risk_array(aligned, score_b)
    n         = len(aligned)

    t_early = np.linspace(max(30.0, 30.0), t_early_end_d, n_pts)
    t_late  = np.linspace(t_late_start_d, t_late_end_d, n_pts)

    def calc_ratio(ev, tm, risk):
        try:
            y = Surv.from_arrays(event=ev, time=tm)
            _, ia_e = cumulative_dynamic_auc(y, y, risk, t_early)
            _, ia_l = cumulative_dynamic_auc(y, y, risk, t_late)
            if ia_e <= 0.5:
                return np.nan
            return float(ia_l / ia_e), float(ia_e), float(ia_l)
        except Exception:
            return np.nan

    res_a = calc_ratio(event_arr, time_arr, a_risk)
    res_b = calc_ratio(event_arr, time_arr, b_risk)
    if isinstance(res_a, float) or isinstance(res_b, float):
        return dict(ratio_a=np.nan, ratio_b=np.nan, iauc_e_a=np.nan,
                    iauc_l_a=np.nan, iauc_e_b=np.nan, iauc_l_b=np.nan,
                    delta=np.nan, ci=(np.nan, np.nan), p=np.nan)

    ratio_a, ia_e_a, ia_l_a = res_a
    ratio_b, ia_e_b, ia_l_b = res_b
    delta_hat = ratio_a - ratio_b

    rng = np.random.default_rng(seed)
    all_idx = rng.integers(0, n, size=(n_boot, n))

    def one_boot(idx):
        try:
            y_b = Surv.from_arrays(event=event_arr[idx], time=time_arr[idx])
            _, ae = cumulative_dynamic_auc(y_b, y_b, a_risk[idx], t_early)
            _, al = cumulative_dynamic_auc(y_b, y_b, a_risk[idx], t_late)
            _, be = cumulative_dynamic_auc(y_b, y_b, b_risk[idx], t_early)
            _, bl = cumulative_dynamic_auc(y_b, y_b, b_risk[idx], t_late)
            if ae <= 0.5 or be <= 0.5:
                return None
            return float(al / ae - bl / be)
        except Exception:
            return None

    deltas = Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(one_boot)(all_idx[i]) for i in range(n_boot)
    )
    deltas = np.array([d for d in deltas if d is not None], dtype=float)
    if deltas.size < max(30, int(0.2 * n_boot)):
        return dict(ratio_a=ratio_a, ratio_b=ratio_b,
                    iauc_e_a=ia_e_a, iauc_l_a=ia_l_a,
                    iauc_e_b=ia_e_b, iauc_l_b=ia_l_b,
                    delta=delta_hat, ci=(np.nan, np.nan), p=np.nan)
    ci_low, ci_high = np.percentile(deltas, [2.5, 97.5])
    p_left  = np.mean(deltas <= 0.0)
    p_right = np.mean(deltas >= 0.0)
    p = float(np.clip(2.0 * min(p_left, p_right), 0.0, 1.0))
    return dict(ratio_a=ratio_a, ratio_b=ratio_b,
                iauc_e_a=ia_e_a, iauc_l_a=ia_l_a,
                iauc_e_b=ia_e_b, iauc_l_b=ia_l_b,
                delta=delta_hat, ci=(float(ci_low), float(ci_high)), p=p)


def analyze_trajectory(aligned, cohort, grid):
    """Module A: iAUC per time window + retention ratio (BV8 vs comparators)."""
    rows_iauc = []
    max_d = np.quantile(aligned[time_col].to_numpy(), grid["q_end"])

    score_order = [(MAIN_LABEL, MAIN_COL)] + COMPARATORS
    for label, col in score_order:
        if col not in aligned.columns:
            continue
        for w_label, ws, we in TIME_WINDOWS:
            t_s = ws * 365.25
            t_e = max_d if we is None else min(we * 365.25, max_d)
            if t_e <= t_s + 60:
                continue
            iauc = compute_window_iauc(aligned, col, t_s, t_e, grid["n_per_window"])
            rows_iauc.append(dict(
                Cohort=cohort, Score=label, ScoreCol=col,
                Window=w_label, WindowStart_y=ws,
                WindowEnd_y=(we if we is not None else round(max_d / 365.25, 2)),
                iAUC=iauc,
            ))
            if not np.isnan(iauc):
                print(f"    [{label:12s} | {w_label:6s}] iAUC = {iauc:.4f}")

    # Retention ratio: early = 0-3y, late = 5-min(10y, max)
    t_early_end_d  = EARLY_ONSET_CUTOFF * 365.25
    t_late_start_d = 5 * 365.25
    t_late_end_d   = min(10 * 365.25, max_d)

    rows_ret = []
    if t_late_end_d > t_late_start_d + 60:
        for label, col in COMPARATORS:
            if col not in aligned.columns:
                continue
            res = paired_retention_ratio(
                aligned, MAIN_COL, col,
                t_early_end_d, t_late_start_d, t_late_end_d,
                n_pts=grid["n_per_window"], n_boot=N_BOOT,
                seed=RANDOM_SEED, n_jobs=N_JOBS,
            )
            rows_ret.append(dict(
                Cohort=cohort, Comparator=label, ComparatorCol=col,
                iAUC_early_BV8=res["iauc_e_a"],
                iAUC_late_BV8=res["iauc_l_a"],
                Retention_BV8=res["ratio_a"],
                iAUC_early_comp=res["iauc_e_b"],
                iAUC_late_comp=res["iauc_l_b"],
                Retention_comp=res["ratio_b"],
                Delta_Retention=res["delta"],
                CI_low=res["ci"][0], CI_high=res["ci"][1],
                p_raw=res["p"],
            ))
            if not np.isnan(res["delta"]):
                print(f"    Retention BV8({res['ratio_a']:.3f}) vs {label}"
                      f"({res['ratio_b']:.3f}): D={res['delta']:+.3f} "
                      f"[{res['ci'][0]:+.3f},{res['ci'][1]:+.3f}] p={res['p']:.4g}")

    return pd.DataFrame(rows_iauc), pd.DataFrame(rows_ret)


def draw_trajectory(pdf, iauc_df, ret_df, cohort):
    """Draw iAUC trajectory line plot + retention ratio bar chart."""
    sub = iauc_df[iauc_df["Cohort"] == cohort]
    if sub.empty:
        return
    windows = [w for w, _, _ in TIME_WINDOWS if w in sub["Window"].unique()]
    scores  = [MAIN_LABEL] + [lab for lab, _ in COMPARATORS]
    scores  = [s for s in scores if s in sub["Score"].unique()]
    data = (sub.pivot(index="Window", columns="Score", values="iAUC")
               .reindex(index=windows, columns=scores))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                             gridspec_kw={"width_ratios": [1.8, 1]})

    # Left: iAUC line plot across time windows
    ax = axes[0]
    x = np.arange(len(windows))
    for s in scores:
        if s not in data.columns:
            continue
        is_main = (s == MAIN_LABEL)
        ax.plot(x, data[s].values, marker="o",
                color=COLORS.get(s, "#888"),
                linewidth=2.8 if is_main else 1.5,
                markersize=9 if is_main else 6,
                linestyle="-" if is_main else "--",
                label=s, zorder=5 if is_main else 3)
    ax.set_xticks(x)
    ax.set_xticklabels(windows)
    ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5)
    ax.set_xlabel("Time window")
    ax.set_ylabel("iAUC within window")
    ax.set_title(f"{cohort} — iAUC trajectory across time windows",
                 fontsize=12, fontweight="bold")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9, ncol=2, loc="lower right")

    # Right: retention ratio bar chart
    ax = axes[1]
    sub_ret = ret_df[ret_df["Cohort"] == cohort].copy()
    if not sub_ret.empty:
        labels = [MAIN_LABEL] + list(sub_ret["Comparator"])
        ratios = [sub_ret["Retention_BV8"].iloc[0]] + list(sub_ret["Retention_comp"])
        colors_b = [COLORS.get(s, "#888") for s in labels]
        ax.bar(labels, ratios, color=colors_b, edgecolor="black", linewidth=0.4)
        ax.axhline(1.0, color="red", linestyle="--", linewidth=1,
                   label="No decay (ratio=1)")
        ax.set_ylabel("Retention = iAUC(5-10y) / iAUC(0-3y)")
        ax.set_title("Long-term retention\n(higher = better preclinical detection)",
                     fontsize=11)
        ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
        ax.grid(axis="y", alpha=0.3)
        ax.legend(fontsize=8)

    fig.tight_layout()
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


# ========================= Module B: Top-10% Event Timing =========================

def analyze_preclinical_timing(aligned, cohort):
    """Compute event timing distribution within top-10% risk group.
    If BV8's top-10% has many events at 5-10y+, it identifies preclinical cases.
    """
    rows = []
    time_arr  = aligned[time_col].to_numpy(dtype=float)
    event_arr = aligned[event_col].to_numpy(dtype=int)
    score_order = [(MAIN_LABEL, MAIN_COL)] + COMPARATORS

    for label, col in score_order:
        if col not in aligned.columns:
            continue
        risk = risk_array(aligned, col)
        thr  = np.quantile(risk, 0.90)
        mask = risk >= thr
        n_top = int(mask.sum())
        if n_top < 30:
            continue

        t_top = time_arr[mask]
        e_top = event_arr[mask]
        n_events_total = int(e_top.sum())
        if n_events_total < 5:
            continue

        # KM-based CIF per window
        try:
            ts, ss = kaplan_meier_estimator(e_top.astype(bool), t_top)
        except Exception:
            continue

        def cif_at(t_day):
            if len(ts) == 0:
                return 0.0
            idx = np.searchsorted(ts, t_day, side="right") - 1
            if idx < 0:
                return 0.0
            return 1.0 - float(ss[idx])

        prev_cif = 0.0
        for w_label, ws, we in TIME_WINDOWS:
            if we is None:
                t_end_d = t_top.max()
                we_disp = round(t_end_d / 365.25, 2)
            else:
                t_end_d = we * 365.25
                we_disp = we
            cif_e = cif_at(t_end_d)
            incident = max(cif_e - prev_cif, 0.0)
            rows.append(dict(
                Cohort=cohort, Score=label, ScoreCol=col,
                Window=w_label, WindowStart_y=ws, WindowEnd_y=we_disp,
                N_top10=n_top,
                CIF_cumulative=cif_e,
                Window_incident_CIF=incident,
                CIF_relative=incident / cif_e if cif_e > 0 else np.nan,
            ))
            prev_cif = cif_e

    return pd.DataFrame(rows)


def draw_preclinical_timing(pdf, df, cohort):
    """Draw grouped bar chart of incident CIF per time window for top-10% group."""
    sub = df[df["Cohort"] == cohort]
    if sub.empty:
        return
    windows = [w for w, _, _ in TIME_WINDOWS if w in sub["Window"].unique()]
    scores  = [MAIN_LABEL] + [lab for lab, _ in COMPARATORS]
    scores  = [s for s in scores if s in sub["Score"].unique()]
    data = (sub.pivot(index="Window", columns="Score", values="Window_incident_CIF")
               .reindex(index=windows, columns=scores))

    fig, ax = plt.subplots(figsize=(10, 5.5))
    x = np.arange(len(windows))
    w = 0.8 / max(len(scores), 1)
    for i, s in enumerate(scores):
        vals = data[s].values if s in data.columns else [np.nan] * len(windows)
        ax.bar(x + i * w - 0.4 + w / 2, vals, w,
               color=COLORS.get(s, "#888"), label=s,
               edgecolor="black", linewidth=0.3)
    ax.set_xticks(x)
    ax.set_xticklabels(windows)
    ax.set_xlabel("Time window after baseline")
    ax.set_ylabel("Incident CIF within window\n(top 10% risk group)")
    ax.set_title(f"{cohort} — Event distribution in top-10% group\n"
                 f"(bars rightward = preclinical identification)",
                 fontsize=12, fontweight="bold")
    ax.legend(fontsize=8, ncol=3, loc="upper right")
    ax.grid(axis="y", alpha=0.3)
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


# ========================= Module C: Sensitivity by Onset Time =========================

def analyze_sensitivity_by_onset(aligned, cohort):
    """Compute sensitivity at top 10%/20% for early, mid, and late onset cases.
    Higher sens_late for BV8 = evidence of preclinical detection.
    """
    rows = []
    time_arr  = aligned[time_col].to_numpy(dtype=float)
    event_arr = aligned[event_col].to_numpy(dtype=int).astype(bool)
    time_years = time_arr / 365.25

    early_mask = event_arr & (time_years <= EARLY_ONSET_CUTOFF)
    late_mask  = event_arr & (time_years > LATE_ONSET_CUTOFF)
    mid_mask   = event_arr & (time_years > EARLY_ONSET_CUTOFF) & \
                 (time_years <= LATE_ONSET_CUTOFF)

    n_early = int(early_mask.sum())
    n_late  = int(late_mask.sum())
    n_mid   = int(mid_mask.sum())
    print(f"    Event distribution: early (<={EARLY_ONSET_CUTOFF}y)={n_early}, "
          f"mid ({EARLY_ONSET_CUTOFF}-{LATE_ONSET_CUTOFF}y)={n_mid}, "
          f"late (>{LATE_ONSET_CUTOFF}y)={n_late}")

    if n_early < 10 and n_late < 10:
        print("    [Skip] insufficient early/late events")
        return pd.DataFrame(rows)

    score_order = [(MAIN_LABEL, MAIN_COL)] + COMPARATORS
    for label, col in score_order:
        if col not in aligned.columns:
            continue
        risk = risk_array(aligned, col)
        for q in TOP_QUANTILES:
            thr = np.quantile(risk, 1 - q)
            flagged = risk >= thr
            rows.append(dict(
                Cohort=cohort, Score=label, ScoreCol=col,
                Top_Quantile=q,
                N_early_cases=n_early,
                N_mid_cases=n_mid,
                N_late_cases=n_late,
                Sens_early=float((flagged & early_mask).sum() / n_early) if n_early else np.nan,
                Sens_mid=float((flagged & mid_mask).sum() / n_mid) if n_mid else np.nan,
                Sens_late=float((flagged & late_mask).sum() / n_late) if n_late else np.nan,
            ))

    df = pd.DataFrame(rows)
    if not df.empty:
        t = df[df["Top_Quantile"] == 0.10]
        print("\n    Sensitivity @ top 10%:")
        print(t[["Score", "Sens_early", "Sens_mid", "Sens_late"]].to_string(index=False))
    return df


def draw_sensitivity_by_onset(pdf, df, cohort):
    """Draw grouped bar chart comparing sensitivity for early/mid/late cases."""
    sub = df[(df["Cohort"] == cohort) & (df["Top_Quantile"] == 0.10)]
    if sub.empty:
        return
    scores = [MAIN_LABEL] + [lab for lab, _ in COMPARATORS]
    sub = sub.set_index("Score").reindex(scores).reset_index().dropna(subset=["Sens_late"])

    fig, ax = plt.subplots(figsize=(10, 5.5))
    x = np.arange(len(sub))
    w = 0.26
    ax.bar(x - w, sub["Sens_early"], w,
           label=f"Early cases (<={EARLY_ONSET_CUTOFF}y)",
           color="#F4A261", edgecolor="black", linewidth=0.3)
    ax.bar(x, sub["Sens_mid"], w,
           label=f"Mid cases ({EARLY_ONSET_CUTOFF}-{LATE_ONSET_CUTOFF}y)",
           color="#A8DADC", edgecolor="black", linewidth=0.3)
    ax.bar(x + w, sub["Sens_late"], w,
           label=f"Late cases (>{LATE_ONSET_CUTOFF}y, preclinical)",
           color="#457B9D", edgecolor="black", linewidth=0.3)
    ax.set_xticks(x)
    ax.set_xticklabels(sub["Score"], rotation=25, ha="right")
    ax.set_ylabel("Sensitivity @ top 10%")
    ax.set_title(f"{cohort} — Sensitivity by case onset time\n"
                 f"(higher 'late cases' bar = better preclinical identification)",
                 fontsize=12, fontweight="bold")
    ax.legend(fontsize=9, loc="upper right")
    ax.grid(axis="y", alpha=0.3)
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


# ========================= Main =========================

def main():
    df = pd.read_csv(DATA_PATH, low_memory=False)
    print(f"Loaded data: {len(df)} rows")

    score_cols = [MAIN_COL] + [c for _, c in COMPARATORS]
    df = coerce_numeric(df, [time_col, event_col] + score_cols)
    df = df.dropna(subset=[time_col, event_col, cohort_col])
    df = df[df[time_col] > 0].copy()

    # Auto-detect time unit and convert years to days if needed
    print("\nTime unit detection:")
    for cohort in df[cohort_col].dropna().unique():
        mask     = df[cohort_col] == cohort
        max_time = df.loc[mask, time_col].max()
        if max_time < 50:
            df.loc[mask, time_col] = df.loc[mask, time_col] * 365.25
            print(f"  {cohort}: max={max_time:.2f} -> years, converted to days")
        else:
            print(f"  {cohort}: max={max_time:.2f} -> already in days")

    preferred = ["UKB", "ELSA", "SHARE", "HRS"]
    cohorts_present = df[cohort_col].unique().tolist()
    cohorts = [c for c in preferred if c in cohorts_present] + \
              sorted([c for c in cohorts_present if c not in preferred])
    print(f"\nCohorts: {cohorts}")

    all_iauc, all_ret, all_timing, all_sens = [], [], [], []
    pdf_A = PdfPages(FIG_DIR / "fig_A_AUC_trajectory.pdf")
    pdf_B = PdfPages(FIG_DIR / "fig_B_preclinical_timing.pdf")
    pdf_C = PdfPages(FIG_DIR / "fig_C_sensitivity_by_onset.pdf")

    try:
        for cohort in cohorts:
            df_c = df[df[cohort_col] == cohort].copy()
            aligned, valid = get_aligned(df_c, score_cols)
            n, e = len(aligned), int(aligned[event_col].sum())
            print(f"\n{'=' * 60}\n> {cohort}  aligned n={n}, events={e}\n{'=' * 60}")
            if n < 200 or e < 30:
                print("  [Skip]"); continue
            grid = COHORT_GRID.get(cohort, DEFAULT_GRID)

            print(f"\n[A] iAUC trajectory & retention:")
            iauc_df, ret_df = analyze_trajectory(aligned, cohort, grid)
            if not iauc_df.empty: all_iauc.append(iauc_df)
            if not ret_df.empty:  all_ret.append(ret_df)
            draw_trajectory(pdf_A, iauc_df, ret_df, cohort)

            print(f"\n[B] Preclinical timing in top 10%:")
            timing_df = analyze_preclinical_timing(aligned, cohort)
            if not timing_df.empty: all_timing.append(timing_df)
            draw_preclinical_timing(pdf_B, timing_df, cohort)

            print(f"\n[C] Sensitivity by onset time:")
            sens_df = analyze_sensitivity_by_onset(aligned, cohort)
            if not sens_df.empty: all_sens.append(sens_df)
            draw_sensitivity_by_onset(pdf_C, sens_df, cohort)
    finally:
        pdf_A.close(); pdf_B.close(); pdf_C.close()

    # Save CSVs
    if all_iauc:
        pd.concat(all_iauc, ignore_index=True).to_csv(
            CSV_DIR / "A_window_iAUC.csv", index=False, encoding="utf-8-sig")
        print(f"\nSaved: A_window_iAUC.csv")

    if all_ret:
        ret_all = pd.concat(all_ret, ignore_index=True)
        # Per-cohort BH-FDR correction
        def add_fdr(g):
            p = g["p_raw"].to_numpy()
            valid = ~np.isnan(p)
            p_adj = np.full(len(p), np.nan)
            if valid.any():
                _, p_adj_v, _, _ = multipletests(p[valid], method="fdr_bh")
                p_adj[valid] = p_adj_v
            g = g.copy(); g["p_fdr"] = p_adj
            g["sig"] = np.where(g["p_fdr"] < 0.05, "*", "")
            return g
        ret_all = ret_all.groupby("Cohort", group_keys=False).apply(add_fdr)
        ret_all.to_csv(CSV_DIR / "A_retention_ratio.csv",
                       index=False, encoding="utf-8-sig")
        print(f"Saved: A_retention_ratio.csv  "
              f"(significant: {int((ret_all['sig'] == '*').sum())}/{len(ret_all)})")

    if all_timing:
        pd.concat(all_timing, ignore_index=True).to_csv(
            CSV_DIR / "B_top10_timing.csv", index=False, encoding="utf-8-sig")
        print(f"Saved: B_top10_timing.csv")

    if all_sens:
        sens_all = pd.concat(all_sens, ignore_index=True)
        sens_all.to_csv(CSV_DIR / "C_sensitivity_by_onset.csv",
                        index=False, encoding="utf-8-sig")
        print(f"Saved: C_sensitivity_by_onset.csv")

    print(f"\nDone. Output directory: {OUT_DIR}")


if __name__ == "__main__":
    main()